In [92]:
import pandas as pd
import numpy as np
import panel as pn
import datetime as dt
pn.extension('tabulator')

import hvplot.pandas

In [93]:
df = pd.read_csv('C:/BHCC/Internship CSC INT 299/NWS_Data/hooj_request1/HeatRiskMixtape.csv')

In [94]:
df

,YEAR,GHCN,NAME,STATE,LATITUDE,LONGITUDE,MaxT_Records,MinT_Records,MaxT_Obs,MinT_Obs,HeatRisk
0,2005.0,USC00020080,AJO,AZ,32.370,-112.860,100.0,73.0,85.0,63.0,1.0
1,2006.0,USC00020080,AJO,AZ,32.370,-112.860,100.0,74.0,86.0,63.0,1.0
2,2007.0,USC00020080,AJO,AZ,32.370,-112.860,100.0,73.0,88.0,63.0,1.0
3,2008.0,USC00020080,AJO,AZ,32.370,-112.860,98.0,72.0,84.0,60.0,1.0
4,2009.0,USC00020080,AJO,AZ,32.370,-112.860,104.0,77.0,92.0,67.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
1139,NaN,USC00299820,WOLF CANYON,NM,35.948,-106.747,NaN,NaN,NaN,NaN,NaN
1140,NaN,USC00420819,BOUNTIFUL BENCH,UT,40.891,-111.850,NaN,NaN,NaN,NaN,NaN
1141,NaN,USC00427064,PROVO BYU,UT,40.246,-111.651,NaN,NaN,NaN,NaN,NaN
1142,NaN,USC00455840,NEWHALEM,WA,48.676,-121.242,NaN,NaN,NaN,NaN,NaN


In [95]:
df.columns

Index(['YEAR', 'GHCN', 'NAME', 'STATE', 'LATITUDE', 'LONGITUDE',
       'MaxT_Records', 'MinT_Records', 'MaxT_Obs', 'MinT_Obs', 'HeatRisk'],
      dtype='object')

In [96]:
# remove duplicates
df.drop_duplicates(inplace = True)
df

,YEAR,GHCN,NAME,STATE,LATITUDE,LONGITUDE,MaxT_Records,MinT_Records,MaxT_Obs,MinT_Obs,HeatRisk
0,2005.0,USC00020080,AJO,AZ,32.370,-112.860,100.0,73.0,85.0,63.0,1.0
1,2006.0,USC00020080,AJO,AZ,32.370,-112.860,100.0,74.0,86.0,63.0,1.0
2,2007.0,USC00020080,AJO,AZ,32.370,-112.860,100.0,73.0,88.0,63.0,1.0
3,2008.0,USC00020080,AJO,AZ,32.370,-112.860,98.0,72.0,84.0,60.0,1.0
4,2009.0,USC00020080,AJO,AZ,32.370,-112.860,104.0,77.0,92.0,67.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
1139,NaN,USC00299820,WOLF CANYON,NM,35.948,-106.747,NaN,NaN,NaN,NaN,NaN
1140,NaN,USC00420819,BOUNTIFUL BENCH,UT,40.891,-111.850,NaN,NaN,NaN,NaN,NaN
1141,NaN,USC00427064,PROVO BYU,UT,40.246,-111.651,NaN,NaN,NaN,NaN,NaN
1142,NaN,USC00455840,NEWHALEM,WA,48.676,-121.242,NaN,NaN,NaN,NaN,NaN


In [97]:
# Make df Pipeline Interactive
idf = df.interactive()

In [98]:
# # Define Panel widgets
year_slider = pn.widgets.IntSlider(name='Year slider', start=2005, end=2025, step=1, value=2005)
year_slider

IntSlider(end=2025, name='Year slider', start=2005, value=2005)

In [99]:
# RadioButtons for MaxT from different impact level
yaxis_MaxTemp = pn.widgets.RadioButtonGroup(
      name='Y axis',
      options = ['MaxT_Records','MaxT_Obs',],
      button_type = 'success'
)

In [100]:
stations = ['AJO', 'CANYON DE CHELLY', 'FORT VALLEY', 'PRESCOTT', 'ROOSEVELT 1 S']

In [101]:
MaxTemp_pipeline = (
    idf[(idf.YEAR <= year_slider) & (idf.NAME.isin(stations))]
    .groupby(['NAME', 'YEAR'])[yaxis_MaxTemp]
    .mean()
    .to_frame()
    .reset_index()
    .sort_values(by='YEAR')
    .reset_index(drop=True)
)

In [102]:
MaxTemp_pipeline

In [103]:
MaxTemp_plot = MaxTemp_pipeline.hvplot(x = 'YEAR', by='NAME', y=yaxis_MaxTemp,line_width=2, title="Maximum Temperature Records in AZ")
MaxTemp_plot

In [104]:
MaxTemp_table = MaxTemp_pipeline.pipe(pn.widgets.Tabulator, pagination='remote', page_size = 10, sizing_mode='stretch_width') 
MaxTemp_table

In [105]:
MaxTemp_vs_MinTemp_scatterplot_pipeline = (
    idf[
        (idf.YEAR == year_slider) &
        (idf.NAME.isin(stations))
    ]
    .groupby(['NAME', 'YEAR', 'MinT_Records'])['MaxT_Records'].mean()
    .to_frame()
    .reset_index()
    .sort_values(by='YEAR')  
    .reset_index(drop=True)
)

In [106]:
MaxTemp_vs_MinTemp_scatterplot_pipeline

In [107]:
MaxTemp_vs_MinTemp_scatterplot = MaxTemp_vs_MinTemp_scatterplot_pipeline.hvplot(x='MinT_Records', 
                                                                y='MaxT_Records', 
                                                                by='NAME', 
                                                                size=80, kind="scatter", 
                                                                alpha=0.7,
                                                                legend=False, 
                                                                height=500, 
                                                                width=500)

In [108]:
MaxTemp_vs_MinTemp_scatterplot

In [109]:
MaxTemp_bar_plot = MaxTemp_pipeline.hvplot(kind='bar', 
                                                     x='NAME', 
                                                     y=yaxis_MaxTemp, 
                                                     title='Maxtemp Bar in AZ')
MaxTemp_bar_plot

In [110]:
#Layout using Template
template = pn.template.FastListTemplate(
    title='HeatRisk in Arizona dashboard', 
    sidebar=[pn.pane.Markdown("#Maximum Temperature Records in AZ"), 
             pn.pane.Markdown("####It shows the correlation between maximum temperature and minimum temperature in AZ"), 
             pn.pane.PNG('NWS.png', sizing_mode='scale_both'),
             pn.pane.Markdown("## Settings"),   
             year_slider],
    main=[pn.Row(pn.Column(yaxis_MaxTemp, 
                           MaxTemp_plot.panel(width=700), margin=(0,25)), 
                 MaxTemp_table.panel(width=500)), 
          pn.Row(pn.Column(MaxTemp_vs_MinTemp_scatterplot.panel(width=600), margin=(0,25)), 
                 pn.Column(MaxTemp_bar_plot.panel(width=600)))],
    accent_base_color="#00FFFF",
    header_background="#008080",
)
# template.show()
template.servable();